<a href="https://colab.research.google.com/github/GustavoTriaquim/Estrutura-de-dados-nao-lineares/blob/main/AULA04/Aula04_E02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx folium -q

import osmnx as ox
import networkx as nx
import folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
def rota_multipla(G, mapa, pontos_nomes, cores):
  coords = [ox.geocode(p) for p in pontos_nomes]
  nos = [ox.distance.nearest_nodes(G, X=c[1], Y=c[0]) for c in coords]

  for i in range(len(nos) - 1):
    rota = nx.shortest_path(G, source=nos[i], target=nos[i+1], weight="lenght")
    pontos = [[G.nodes[n]['y'], G.nodes[n]['x']] for n in rota]

    folium.PolyLine(
        locations=pontos,
        color=cores[i % len(cores)],
        wieght=4,
        opacity=0.85,
        tootltip=f"{pontos_nomes[i]} -> {pontos_nomes[i+1]}"
    ).add_to(mapa)

  for nome, c in zip(pontos_nomes, coords):
    folium.Marker(c, popup=nome).add_to(mapa)

G_sjp = ox.graph_from_place(
    "São José dos Pinhais, Paraná, Brasil",
    network_type="drive",
    truncate_by_edge=True,
    retain_all=True,
    simplify=True
)

mapa_2_1 = folium.Map(
    location=ox.geocode("São José dos Pinhais, Paraná, Brasil"),
    zoom_start=12
)

trajeto_1 = [
    "Parque da Fonte, São José dos Pinhais, PR", "Afonso Pena, São José dos Pinhais, PR",
    "Cidade Jardim, São José dos Pinhais, PR", "Centro, São José dos Pinhais, PR",
    "São Pedro, São José dos Pinhais, PR"
]
rota_multipla(G_sjp, mapa_2_1, trajeto_1, ["blue", "green", "orange", "purple"])

trajeto_2 = [
    "São Marcos, São José dos Pinhais, PR", "Barro Preto, São José dos Pinhais, PR",
    "Del Rey, São José dos Pinhais, PR", "Pedro Moro, São José dos Pinhais, PR",
    "Quississana, São José dos Pinhais, PR"
]
rota_multipla(G_sjp, mapa_2_1, trajeto_2, ["red", "brown", "darkblue", "black"])

mapa_2_1

In [ ]:
def geocode_seguro(nome, tentativas_extra=None):
    """Tenta geocodificar o nome; se falhar, tenta variações alternativas."""
    candidatos = [nome] + (tentativas_extra or [])
    for candidato in candidatos:
        try:
            return ox.geocode(candidato)
        except Exception:
            continue
    raise ValueError(f"Não foi possível geocodificar nenhuma variação de: {nome}")

def rota_no_mapa(G, mapa, origem_nome, destino_nome, cor="blue",
                  origem_alt=None, destino_alt=None):
    origem_coord = geocode_seguro(origem_nome, origem_alt)
    destino_coord = geocode_seguro(destino_nome, destino_alt)
    origem_no = ox.distance.nearest_nodes(G, X=origem_coord[1], Y=origem_coord[0])
    destino_no = ox.distance.nearest_nodes(G, X=destino_coord[1], Y=destino_coord[0])
    rota = nx.shortest_path(G, source=origem_no, target=destino_no, weight="length")
    pontos = [[G.nodes[n]['y'], G.nodes[n]['x']] for n in rota]
    folium.PolyLine(locations=pontos, color=cor, weight=4, opacity=0.85,
                     tooltip=f"{origem_nome} → {destino_nome}").add_to(mapa)
    folium.Marker(pontos[0], popup=origem_nome, icon=folium.Icon(color="green")).add_to(mapa)
    folium.Marker(pontos[-1], popup=destino_nome, icon=folium.Icon(color="red")).add_to(mapa)
    return rota

cidades_rmc = [
    "Curitiba, Paraná, Brasil", "Colombo, Paraná, Brasil",
    "Fazenda Rio Grande, Paraná, Brasil", "Pinhais, Paraná, Brasil",
    "Campo Largo, Paraná, Brasil", "Araucária, Paraná, Brasil"
]

G_rmc = ox.graph_from_place(
    cidades_rmc,
    network_type="drive",
    truncate_by_edge=True,
    retain_all=True,
    simplify=True
)

mapa_2_2 = folium.Map(location=ox.geocode("Curitiba, Paraná, Brasil"), zoom_start=10)

rota_no_mapa(G_rmc, mapa_2_2, "Terminal Colombo, Colombo, PR", "Terminal Fazenda Rio Grande, Fazenda Rio Grande, PR", "blue")
rota_no_mapa(G_rmc, mapa_2_2, "Terminal Pinhais, Pinhais, PR", "Terminal Campo Largo, Campo Largo, PR", "red")
rota_no_mapa(G_rmc, mapa_2_2, "Terminal Araucária, Araucária, PR", "Terminal Santa Felicidade, Curitiba, PR", "green")

mapa_2_2